## Lecture 9: Testing & Documentation

### ****Exercise 1:** Extract and Test**

****Goal:** find a hard-to-test function in your own code; extract the computation core as a pure function; write one test.**

I was unable to locate any hard-to-test functions in my own code, so I have used the code form side number 19 instead (as indicated by step 2).

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

# From slide number 19

def run_mandelbrot(N, max_iter):
    t0 = time.perf_counter()
    result = []
    for i in range(N):
        for j in range(N):
            z = 0j
            c = x_min + j*(x_max-x_min)/N + 1j*(y_min + i*(y_max-y_min)/N)
            for n in range(max_iter):
                if abs(z) > 2:
                    result.append(n); break
                z = z*z + c
            else:
                result.append(max_iter)
    print(f"Time: {time.perf_counter()-t0:.3f}s")
    plt.imshow(np.array(result).reshape(N, N)); plt.show()

#### ****Step 1:** — identify the problem:**

**Does a function mix computation with timing, printing, or plotting? That is the one to fix. The “Untestable Function” slide is the reference.**

Yes, it mixes of the three. The core functionality would be the mandelbrot point computation on line ~ [12-17]


We have multiple problems:
1. `x_min`, `x_max`, `y_min`, and `y_max` are undefined. I suspect that they were likely globals.
2. The use of `;` should be avoided in Python.
2. Half commitment. Use either Numpy or not. for example the results list should be the right shape to begin with.
3. Unable to select performance timing resolution. sec may be too fine for some use cases. This should not be part of the function to begin with. 
4. `(x_max-x_min)/N` and `y_min + i*(y_max-y_min)/N` should only have been computed once. There is not need to recompute them per pixel. This is wasting performance.
5. Nothing is returned, making it very hard to test.
6. Unconditioned print statements. I cannot control if we should print or not. prints should not be part of the function to begin with.
7. Assumption that the image/grid is squared. There is not technical reasoning as to why it must be. The image/grid should not even be part of this function.
8. `plt.show()` assumes graphical session, which may not be the case on servers. Thus the code fails to run. Plotting should not even be part of this function.
9. There are no type hints.
10. Computing the performance timing includes sub-computing/parsing the print, making the timing (slightly) delayed more than it should have been.

To put it simply: it's testable.

#### ****Step 2:** — extract:**

**Pull the inner pixel computation into a standalone function that takes `c` and `max_iter` and returns an `int`**

Slide number 20 done it very nicely, so I am using that.

In [2]:
# From slide 20

def mandelbrot_pixel(c: complex, max_iter: int) -> int:
    z = 0j
    for n in range(max_iter):
        if z.real*z.real + z.imag*z.imag > 4.0:
            return n
        z = z*z + c
    return max_iter

#### ****Step 3:** — test:**

- **If your code is already well-structured, extract mandelbrot pixel from your naive or Numba implementation and test it**

I would consider my code to be (at least to some degree) well-structured.

In [3]:
from numba import njit

# From lecture 3 milestone 3

# From slide 36. Approach A (where I have filled in the gaps in the hybrid)
@njit
def mandelbrot_point_numba(c, max_iter=100):
    z = 0j
    for n in range(max_iter):
        if z.real*z.real+z.imag*z.imag > 4.0:
            return n
        z = z*z + c
    return max_iter

- **Choose a test value you can justify mathematically — not one you read off a plot**

In [4]:
# From slide 20, adapted to be used in a function

def test_ex1_step1():
    # origin: never escapes
    assert mandelbrot_pixel(0+0j, 100) == 100

    # far outside: escapes on iter 1
    assert mandelbrot_pixel(5.0+0j, 100) == 1


def test_ex1_step2():
    assert mandelbrot_point_numba(0+0j, 100) == 100
    assert mandelbrot_point_numba(5.0+0j, 100) == 1

****Done?** Run `pytest -v` and confirm the test passes → discuss what “hard to test” looked like in your code.**

In [5]:
!pytest -v l09_ex1_test.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/conda/envs/nsc/bin/python3.11
cachedir: .pytest_cache
rootdir: /ncs
collected 2 items                                                              

l09_ex1_test.py::test_ex1_step1 PASSED                                   [ 50%]
l09_ex1_test.py::test_ex1_step2 PASSED                                   [100%]

============================== 2 passed in 0.42s ===============================


### ****Exercise 2:** Add a Docstring**

****Goal:** add a NumPy-style docstring to one Mandelbrot function.**

In [6]:
# From lecture 4 milestone 1

@njit
def mandelbrot_chunk(row_start, row_end, N,
                     x_min, x_max, y_min, y_max, max_iter):
    out = np.empty((row_end - row_start, N), dtype=np.int32)
    dx = (x_max - x_min) / N
    dy = (y_max - y_min) / N
    for r in range(row_end - row_start):
        c_imag = y_min + (r + row_start) * dy
        for col in range(N):
            out[r, col] = mandelbrot_pixel(x_min + col*dx, c_imag, max_iter)
    return out

I have chosen the `mandelbrot_chunk()` function from lecture 4.

**Choose a function that:**

- **Does computation (not just timing or plotting)**

Computes mandelbrot pixels based on chunks, excluding timing and plotting - **✓**

- **Has at least two parameters with non-obvious types or semantics**

The function has more than two parameters, each with no obvious or semantic meaning.

- **Returns a meaningful value**

The function returns the computed mandelbrot set image based on iterations.

**Sections to include (NumPy style):**

In [7]:
# Updated to follow NumPy style
import numpy.typing as npt

# From lecture 4

@njit
def mandelbrot_chunk(
    row_start: int, 
    row_end: int, 
    N: int,
    x_min: float, 
    x_max: float, 
    y_min: float, 
    y_max: float, 
    max_iter: int
) -> npt.NDArray[np.int32]:
    """Computes the mandelbrot set based on a single chunk.
    
    Parameters
    ----------
    row_start : int
        The starting index at which the chunk begins.
    row_end : int
        The end index at which the chunk ends.
    N : int
        Number of columns (image width). Assumes squared result image.
    x_min : float
        Minimum x-value (left boundary) of the complex plain.
    x_max : float
        Maximum x-value (right boundary) of the complex plain.
    y_min : float
        Minimum y-value (bottom boundary) of the complex plain.
    y_max : float
        Maximum y-value (top boundary) of the complex plain.
    max_iter : int
        Maximum number of iterations per pixel.
    
    Returns
    -------
    npt.NDArray[np.int32]:
        2D array with iteration counts per pixel.
    """

    out = np.empty((row_end - row_start, N), dtype=np.int32)
    dx = (x_max - x_min) / N
    dy = (y_max - y_min) / N
    for r in range(row_end - row_start):
        c_imag = y_min + (r + row_start) * dy
        for col in range(N):
            out[r, col] = mandelbrot_pixel(x_min + col*dx, c_imag, max_iter)
    return out

1. **One-line summary sentence (ends with a period)**

Done. - **✓**

2. **Parameters — name, type, and description for each parameter**

Done. - **✓**

3. **Returns — type and description of the return value**

Done. - **✓**

4. **Examples (optional but useful)**

Excluded since it is optional.

****Test your docstring:** read it aloud. Does it tell a new reader everything they need to call the function correctly? Does it state what the return value means, not just its type?**

I have read it aloud. I would say that it does tell a new reader what they need to know, including the return values, their meaning, and types.

****Done?** Discuss with a neighbour — compare style choices and what you chose to document.**

Noted! My peer have chosen a different function, the mandelbrot pixel. They included more of the mathematics while I included a more text descriptive docs.

### ****Exercise 3 (Optional):** Branch Coverage Report**

****Goal:** generate a branch coverage report and identify untested branches.**

```python
# Install if needed:
mamba install pytest pytest-cov
# Run with branch coverage:
pytest --cov=. --cov-branch --cov-report=term-missing -v
```

```bash
pytest --cov=. --cov-branch --cov-report=term-missing   #"." is current directory
```

Okay! I get the following:

In [8]:
!pytest --cov=. --cov-branch --cov-report=term-missing -v

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/conda/envs/nsc/bin/python3.11
cachedir: .pytest_cache
rootdir: /ncs
plugins: cov-7.1.0
collected 2 items                                                              

l09_ex1_test.py::test_ex1_step1 PASSED                                   [ 50%]
l09_ex1_test.py::test_ex1_step2 PASSED                                   [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.11.14-final-0 _______________

Name                         Stmts   Miss Branch BrPart  Cover   Missing
------------------------------------------------------------------------
compile_miniproject.py          18     18      0      0     0%   1-30
compile_mp1.py                  10     10      2      0     0%   1-20
compile_mp2.py                  14     14      2      0     0%   1-30
co

**What to look for:**

- **Lines listed under “missing” — never executed by any test**

Noted! I have a total of 292 misses.

- **Branch arrows such as `3->5` — the `if` whose `else` path was never taken**

I have no branch arrows.

- **In `mandelbrot_pixel`: both sides of the escape condition must be covered — a pixel that escapes and a pixel that does not**

I see no `mandelbrot_pixel` name in the output.

****Done?** Identify one untested branch; write a test that covers it → discuss with a neighbour.**

Since I have no `mandelbrot_pixel()` function and file, lets create it and its tests.

In [9]:
%pycat mandelbrot_pixel.py

# From lecture 9, slide 20

def mandelbrot_pixel(c: complex, max_iter: int) -> int:
    z = 0j
    for n in range(max_iter):
        if z.real*z.real + z.imag*z.imag > 4.0:
            return n
        z = z*z + c
    return max_iter


In [10]:
%pycat test_mandelbrot_pixel.py

# From lecture 9, code_example.md (moodle)

import pytest
from mandelbrot_pixel import mandelbrot_pixel

KNOWN_CASES = [
    (0+0j,    100, 100),   # origin: never escapes
    (5.0+0j,  100,   1),   # far outside, escapes on iteration 1
    (-2.5+0j, 100,   1),   # left tip of set
]

@pytest.mark.parametrize("c, max_iter, expected", KNOWN_CASES)
def test_mandelbrot_pixel(c, max_iter, expected):
    assert mandelbrot_pixel(c, max_iter) == expected


In [11]:
!pytest --cov=mandelbrot_pixel --cov-branch --cov-report=term-missing -v

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/conda/envs/nsc/bin/python3.11
cachedir: .pytest_cache
rootdir: /ncs
plugins: cov-7.1.0
collected 5 items                                                              

l09_ex1_test.py::test_ex1_step1 PASSED                                   [ 20%]
l09_ex1_test.py::test_ex1_step2 PASSED                                   [ 40%]
test_mandelbrot_pixel.py::test_mandelbrot_pixel[0j-100-100] PASSED       [ 60%]
test_mandelbrot_pixel.py::test_mandelbrot_pixel[(5+0j)-100-1] PASSED     [ 80%]
test_mandelbrot_pixel.py::test_mandelbrot_pixel[(-2.5+0j)-100-1] PASSED  [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.11.14-final-0 _______________

Name                  Stmts   Miss Branch BrPart  Cover   Missing
----------------------------------------------------

We now have *100%* code coverage (for `mandelbrot_pixel`)

> NOTE: **ADVANCED EXERCISES** 

```sh
mamba install hypothesis sphinx mutmut
```

### ****Exercise 4 (Optional):** Hypothesis**

****Goal:** Write property tests for two structural invariants of `mandelbrot_pixel`**

#### **Software Quality — Advanced (Optional): Hypothesis**

```python
from hypothesis import given, settings
from hypothesis.strategies import integers

@given(integers())          # generates random integers
def test_abs_non_negative(x):
    assert abs(x) >= 0      # must hold for ANY integer
```

#### **Software Quality — Advanced (Optional): Hypothesis — Mandelbrot**

```python
# Draw random points with |c| <= 3 (covers both inside and outside the set)
@given(complex_numbers(max_magnitude=3.0, allow_nan=False, allow_infinity=False))
@settings(max_examples=200)
def test_result_in_range(c):
    assert 0 <= mandelbrot_pixel(c, 100) <= 100
```

```python
# Draw random points far outside the set (|c| between 3 and 10)
@given(complex_numbers(min_magnitude=3.0, max_magnitude=10.0,
                       allow_nan=False, allow_infinity=False))
def test_outside_set_escapes(c):
    assert mandelbrot_pixel(c, 100) < 100
```

#### **Software Quality — Advanced (Optional): Mutation Testing**

```bash
mamba install mutmut
mutmut run        # inject mutations and run tests
mutmut results    # list surviving mutants
```

```python
assert mandelbrot_pixel(0+2j, 100) == 2   # catches it: mutant returns 1, assertion fails
assert mandelbrot_pixel(0+2j, 100) < 100  # misses it:  mutant returns 1, assertion passes
```

### ****Exercise 5 (Optional):** Performance regression**

****Goal:** Add a performance regression test asserting NumPy is significantly faster than naive**

```python
import time, numpy as np

def time_once(fn, *args):
    t0 = time.perf_counter()
    fn(*args)
    return time.perf_counter() - t0

def test_numpy_faster_than_naive():
    N, MAX_ITER = 64, 100   # small grid: low variance, fast to run
    t_naive = time_once(mandelbrot_naive,  N, MAX_ITER)
    t_numpy = time_once(mandelbrot_numpy,  N, MAX_ITER)
    assert t_numpy < t_naive / 5, (
        f"NumPy ({t_numpy:.4f}s) not 5x faster than naive ({t_naive:.4f}s)"
    )
```

### ****Exercise 6 (Optional):** GitHub Actions CI**

****Goal:** Set up a workflow that runs your test suite automatically on every push**

```yaml
name: Tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install numpy pytest pytest-cov
      - run: pytest --cov=. -v
```

### ****Exercise 7 (Optional):** Sphinx**

****Goal:** Generate HTML documentation from your docstrings**

### ****Exercise 8 (Optional):** mutmut mutation testing**

****Goal:** Run mutation testing; identify surviving mutants; add tests to kill them**